# 04 — Train YOLO26m

Pipeline Ultralytics YOLO26m trên VNTS, 52 lớp, không crop từng biển. Checkpoint chọn theo mAP50-95 trên val. Mosaic và mixup chỉ chạy khi `ONLINE = True`.


## 0) Bootstrap (Kaggle / Colab)

Clone hoặc pull nhánh, rồi cài dependency. Trên Kaggle bật Internet: On.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Detection.git"
REPO_DIR = "VN-Traffic-Sign-Detection"
REPO_BRANCH = "yolo"  # Team YOLO đổi thành yolo. Team Faster R-CNN đổi thành faster-rcnn khi nhánh đã có trên origin.

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

if (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("Using the repo this notebook sits in:", repo_path)
elif not (repo_path / ".git").exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')

git = f'git -C "{repo_path}"'
sh(f"{git} fetch origin {REPO_BRANCH}")
sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())
sh(f"{git} log --oneline -1")

sh('pip install -q -e ".[detect]"')
sh("pip install -q wandb")

import torch
import vn_tsd
from vn_tsd.config.resolve import resolve_config
print(
    f"Bootstrap OK. PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}"
)


## 1) Đường dẫn data/processed

Cache offline dùng chung: ảnh letterbox và nhãn YOLO, không crop từng biển. Trên Kaggle gắn dataset bằng Add Input rồi điền vào `CANDIDATES`.


In [ ]:
from pathlib import Path

# After Add Input on Kaggle, paste the dataset path here.
CANDIDATES = [
    "/kaggle/input/datasets/zero001206/vn-traffic-sign-detection-preprocessed-data/processed",
    #"data/processed",
]

def _ready(path: Path) -> bool:
    return (path / "data.yaml").exists() or (path / "images" / "train").exists()

PROCESSED_ROOT = next((c for c in CANDIDATES if c and _ready(Path(c))), None)
if PROCESSED_ROOT is None:
    print("[!] data/processed chưa có data.yaml hoặc images/train.")
    print("    Chạy preprocessing offline trước, hoặc gắn dataset Kaggle vào CANDIDATES.")
    PROCESSED_ROOT = "data/processed"
else:
    print("[OK] Processed root:", PROCESSED_ROOT)


## 2) Cấu hình

Sửa biến ở đây. Batch là tổng của cả hai GPU, không phải mỗi GPU. Không lật ngang.


In [ ]:
# Batch train và eval batch là tổng hai GPU, và phải giống nhau ở cả hai pipeline.
# Chọn số sát VRAM của 2 T4. Hai pipeline phải dùng cùng tổng.
# Đổi batch, eval batch hoặc early_stopping_patience thì báo team kia, rồi sửa cả notebook còn lại và configs/shared.yaml.
# Các biến ở đây chỉ là mức cơ bản. Khi build pipeline, team có thể thêm biến khác cho dễ tune.
PIPELINE = "yolo26m"
EPOCHS = 50
LR0 = 0.01
BATCH_SIZE = 8
EARLY_STOPPING_PATIENCE = 10
EVAL_BATCH_SIZE = 16
WEIGHTS = "yolo26m.pt"
IMGSZ = 640
ONLINE = False          # True chỉ khi đang tune online preprocess
MOSAIC = True           # chỉ áp dụng khi ONLINE = True
MIXUP = False
REQUIRE_WANDB = True
USE_AMP = True
PRIMARY_METRIC = "mAP50-95"
SEED = 42
# Tên hiện trên W&B: {pipeline}_s{seed}_ep{epochs}_bs{batch}[_online]_{hash8}
suffix = "_online" if ONLINE else ""
import uuid
RUN_NAME = f"{PIPELINE}_s{SEED}_ep{EPOCHS}_bs{BATCH_SIZE}{suffix}_{uuid.uuid4().hex[:8]}"


## 3) Resolve config

Ghép biến ở trên với `configs/shared.yaml` và file pipeline. Ghi file overrides để lệnh torchrun bên dưới dùng đúng cấu hình này.


In [ ]:
from pathlib import Path
OVERRIDES = {
    "seed": SEED,
    "model": {"weights": WEIGHTS, "imgsz": IMGSZ},
    "train": {
        "epochs": EPOCHS,
        "lr0": LR0,
        "batch_size": BATCH_SIZE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "amp": USE_AMP,
        "require_wandb": REQUIRE_WANDB,
        "run_name": RUN_NAME,
        "primary_metric": PRIMARY_METRIC,
    },
    "eval": {"batch_size": EVAL_BATCH_SIZE, "auto_eval_test": True, "split": "test"},
    "online_preprocess": {"enabled": ONLINE, "mosaic": MOSAIC, "mixup": MIXUP, "hflip": False},
    "data": {"root": PROCESSED_ROOT},
}


from vn_tsd.config.resolve import resolve_config
from vn_tsd.utils.io import save_yaml

runtime_yaml = "configs/runtime/kaggle.yaml" if Path("/kaggle/working").exists() else "configs/runtime/local.yaml"
cfg = resolve_config(
    pipeline_yaml=f"configs/pipelines/{PIPELINE}.yaml",
    shared_yaml="configs/shared.yaml",
    runtime_yaml=runtime_yaml,
    overrides=OVERRIDES,
)

OVERRIDES_PATH = Path("outputs") / "_notebook" / f"overrides_{PIPELINE}.yaml"
save_yaml(OVERRIDES, OVERRIDES_PATH)

train = cfg.get("train", {})
print("=== Resolved configuration ===")
print("Run name        :", RUN_NAME)
print("Pipeline        :", cfg.get("pipeline"))
print("Epochs          :", train.get("epochs"))
print("Primary metric  :", train.get("primary_metric"))
print("Batch total     :", train.get("batch_size"))
print("Eval batch total:", cfg.get("eval", {}).get("batch_size"))
print("AMP             :", train.get("amp"))
print("W&B required    :", train.get("require_wandb"))
print("Online          :", cfg.get("online_preprocess", {}).get("enabled"))
print("Data root       :", cfg.get("data", {}).get("root"))
print("Overrides file  :", OVERRIDES_PATH)


## 4) Xác thực W&B
Đọc Secret WANDB_API_KEY; run training được tạo ở cell cuối, không tạo run smoke riêng.
Host lấy từ cfg. Cần key đúng server và quyền ghi entity/project. REQUIRE_WANDB=False
sẽ tắt tracking trong notebook. Không in key ra output.

In [ ]:
import os
import wandb

base_url = cfg['project']['wandb_base_url']
entity = cfg['project']['wandb_entity']
project = cfg['project']['wandb_project']
os.environ['WANDB_BASE_URL'] = base_url
if REQUIRE_WANDB:
    if not os.environ.get('WANDB_API_KEY'):
        from kaggle_secrets import UserSecretsClient
        os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY').strip()
    wandb.login(host=base_url, relogin=True)
    print('W&B authentication ready; training run starts in the last cell.')

## 5) Chuẩn bị dataset rồi train một GPU
Cell dataset dưới đây giữ ảnh/nhãn Input và tạo data.yaml tại Working.
Cell cuối gọi pipeline trực tiếp trong notebook để callback dùng cùng W&B run;
không gọi subprocess/torchrun. Lần đầu đặt EPOCHS=1 ở mục 2 rồi chạy lại mục 3.
W&B ghi train/val loss, validation AP, LR và ảnh kết quả cuối run.
AR100 chưa có nên không gửi số 0. Test metrics chỉ ghi summary sau train.
History được ghi từng epoch. Native Ultralytics W&B callbacks được bỏ riêng cho trainer
để không log trùng hoặc đóng run sớm. Không sửa settings W&B toàn cục.

In [ ]:

from pathlib import Path
import yaml

# ============================================================
# 1. DATASET PATHS
# ============================================================

source_root = Path(PROCESSED_ROOT).resolve()
source_yaml = source_root / "data.yaml"

fixed_root = Path("/kaggle/working/yolo_dataset")
fixed_yaml = fixed_root / "data.yaml"

if not source_yaml.is_file():
    raise FileNotFoundError(source_yaml)

# ============================================================
# 2. VALIDATE DATASET STRUCTURE
# ============================================================

for split in ("train", "val", "test"):
    image_dir = source_root / "images" / split

    if not image_dir.is_dir():
        raise FileNotFoundError(
            f"Missing images directory: {image_dir}"
        )

# ============================================================
# 3. LINK DATASET DIRECTORIES
# ============================================================

fixed_root.mkdir(parents=True, exist_ok=True)

for folder in ("images", "labels"):
    src = source_root / folder
    dst = fixed_root / folder

    if src.exists():
        if dst.is_symlink() and dst.resolve() == src:
            continue
        if dst.exists() or dst.is_symlink():
            raise FileExistsError(
                f"Existing destination: {dst}"
            )
        dst.symlink_to(src, target_is_directory=True)

# ============================================================
# 4. FIX DATA.YAML
# ============================================================

with source_yaml.open("r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = str(fixed_root)
data_cfg["train"] = "images/train"
data_cfg["val"] = "images/val"
data_cfg["test"] = "images/test"

with fixed_yaml.open("w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_cfg,
        f,
        sort_keys=False,
        allow_unicode=True
    )

# ============================================================
# 5. UPDATE NOTEBOOK CONFIG
# ============================================================

PROCESSED_ROOT = str(fixed_root)

print("Dataset configuration fixed.")
print("Data root:", PROCESSED_ROOT)
print("Data YAML:", fixed_yaml)
print(yaml.safe_dump(data_cfg, sort_keys=False))


In [ ]:
from vn_tsd.pipelines.yolo.train import YOLOPipeline
from vn_tsd.runtime.run_dir import make_run_dir
from vn_tsd.runtime.pack import zip_run_dir
from vn_tsd.utils.io import save_json, save_yaml
from vn_tsd.utils.seed import set_seed
from vn_tsd.eval.metrics import summarize_yolo

# The dataset preparation cell changes PROCESSED_ROOT after resolve_config().
cfg['data']['root'] = PROCESSED_ROOT
cfg['runtime_cfg'].update(device=0, gpus=1)
set_seed(cfg['seed'])
run_dir = make_run_dir('/kaggle/working/outputs', PIPELINE, RUN_NAME)

class NotebookYOLOPipeline(YOLOPipeline):
    def _build_train_args(self):
        args = super()._build_train_args()
        args['device'] = 0
        return args

history = []
pending_epochs = set()

def remove_native_wandb(trainer):
    for event, handlers in trainer.callbacks.items():
        trainer.callbacks[event] = [fn for fn in handlers
            if fn.__module__ != 'ultralytics.utils.callbacks.wb']

def mark_epoch(trainer):
    pending_epochs.add(trainer.epoch + 1)

if wandb.run is not None:
    raise RuntimeError('Finish the existing W&B run before starting another.')
run = None
success = False
try:
    if REQUIRE_WANDB:
        run = wandb.init(project=project, entity=entity, name=RUN_NAME,
                         config=cfg, dir=str(run_dir), mode='online', job_type='train')
        run.define_metric('epoch')
        run.define_metric('train/*', step_metric='epoch')
        run.define_metric('val/*', step_metric='epoch')
        run.define_metric('lr/*', step_metric='epoch')
        save_json({'id': run.id, 'url': run.url}, run_dir / 'wandb_run.json')
        print('W&B training charts:', run.url)

    def log_epoch(trainer):
        epoch = trainer.epoch + 1
        # Ignore the extra best-checkpoint validation after training ends.
        if epoch not in pending_epochs:
            return
        pending_epochs.remove(epoch)
        scores = summarize_yolo(trainer.validator.metrics)
        values = {'epoch': epoch}
        values.update({k: float(v) for k, v in
                       trainer.label_loss_items(trainer.tloss, prefix='train').items()})
        values.update({k: float(v) for k, v in trainer.metrics.items()})
        values.update({f'val/{k}': v for k, v in scores.items() if v is not None})
        values.update({f'lr/{k}': float(v) for k, v in trainer.lr.items()})
        history.append({**values, 'val/AR100': scores['AR100']})
        save_json(history, run_dir / 'history.json')
        if run is not None:
            run.log(values)

    callbacks = {
        'on_pretrain_routine_start': [remove_native_wandb],
        'on_train_epoch_end': [mark_epoch],
        'on_fit_epoch_end': [log_epoch],
    }
    metrics = NotebookYOLOPipeline(cfg, run_dir).fit(callbacks=callbacks)
    if run is not None:
        run.summary.update({f'test/{k}': v for k, v in metrics['test_metrics'].items()
                            if v is not None})
        for filename in ('results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png'):
            path = run_dir / 'train' / filename
            if path.is_file():
                run.log({f'figures/{path.stem}': wandb.Image(str(path))})
    success = True
finally:
    if run is not None:
        run.finish(exit_code=0 if success else 1)

print('Metrics:', metrics)
print('ZIP:', zip_run_dir(run_dir))